# Clustering Jerárquico — Ames Housing Dataset (2006–2024)
**Objetivo:** Aplicar Clustering Jerárquico con múltiples tipos de linkage, determinar el número óptimo de clústers mediante Silhouette Score, visualizar con PCA y t-SNE, y comparar los resultados con K-Means.

## 0. Importaciones y configuración

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')
PALETTE = sns.color_palette('tab10')

df_raw = pd.read_csv('ames_combined_2006_2024.csv', low_memory=False)
print(f'Shape original: {df_raw.shape}')
df_raw.head(3)

---
## 1. Preparación del Dataset
### 1.1 Selección de variables numéricas relevantes

In [ ]:
NUM_FEATURES = [
    'OverallQual', 'OverallCond', 'GrLivArea', 'TotalBsmtSF',
    'GarageArea', 'YearBuilt', 'LotArea', 'SalePrice',
    'TotRmsAbvGrd', 'Fireplaces', '1stFlrSF', 'GarageCars'
]

df = df_raw[NUM_FEATURES].dropna().reset_index(drop=True)

# Eliminar outlier del clúster sintético detectado en K-Means (SalePrice constante $42M)
df = df[df['SalePrice'] < 1_000_000].reset_index(drop=True)

print(f'Shape después de limpieza: {df.shape}')
print(f'Rango SalePrice: ${df["SalePrice"].min():,} — ${df["SalePrice"].max():,}')
df.describe().round(1)

### 1.2 Escalado y muestra para clustering jerárquico

In [ ]:
scaler = StandardScaler()
X_full = scaler.fit_transform(df)

# El clustering jerárquico es O(n²) en memoria — usamos muestra estratificada
# El dendrograma y linkage se hacen sobre esta muestra
N_SAMPLE = 2000
idx_sample = np.random.choice(len(X_full), N_SAMPLE, replace=False)
X = X_full[idx_sample]
df_sample = df.iloc[idx_sample].reset_index(drop=True)

print(f'Escalado con StandardScaler ✔')
print(f'Muestra para jerárquico: {N_SAMPLE} observaciones')
print(f'Dataset completo para PCA/t-SNE: {len(X_full)} observaciones')

---
## 2. Aplicación del Clustering Jerárquico
### 2.1 Comparación de linkages: Ward, Complete, Average

In [ ]:
LINKAGES = ['ward', 'complete', 'average']
LINKAGE_LABELS = {
    'ward':     'Ward (minimiza varianza intra-clúster)',
    'complete': 'Complete (maximiza distancia inter-clúster)',
    'average':  'Average (distancia promedio entre pares)'
}
LINKAGE_COLORS = ['steelblue', 'darkorange', 'forestgreen']

Z = {}  # matrices de linkage
for method in LINKAGES:
    metric = 'euclidean' if method == 'ward' else 'euclidean'
    Z[method] = linkage(X, method=method, metric='euclidean')
    print(f'{method:10s}: linkage calculado ✔  |  forma: {Z[method].shape}')

### 2.2 Dendrogramas comparativos (Ward, Complete, Average)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Dendrogramas comparativos — Clustering Jerárquico', fontsize=14, y=1.01)

for ax, method, color in zip(axes, LINKAGES, LINKAGE_COLORS):
    dendrogram(
        Z[method],
        ax=ax,
        truncate_mode='lastp',
        p=30,
        leaf_rotation=90,
        leaf_font_size=8,
        show_contracted=True,
        link_color_func=lambda k: color
    )
    ax.set_title(f'Linkage: {method.capitalize()}', fontsize=12)
    ax.set_xlabel('Observaciones (truncado)')
    ax.set_ylabel('Distancia')

plt.tight_layout()
plt.savefig('hier_fig1_dendrogramas.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig1_dendrogramas.png')

---
## 3. Evaluación del Número de Clústers
### 3.1 Silhouette Score para k = 2 … 10 (linkage Ward)

In [ ]:
K_RANGE = range(2, 11)
results = {}

for method in LINKAGES:
    sil_scores = []
    for k in K_RANGE:
        labels = fcluster(Z[method], k, criterion='maxclust')
        score = silhouette_score(X, labels, sample_size=1500, random_state=RANDOM_STATE)
        sil_scores.append(round(score, 4))
    results[method] = sil_scores
    best_k = list(K_RANGE)[sil_scores.index(max(sil_scores))]
    print(f'{method:10s}: scores = {sil_scores}  →  k óptimo = {best_k} (Silhouette = {max(sil_scores):.4f})')

sil_df = pd.DataFrame(results, index=list(K_RANGE))
sil_df.index.name = 'k'
sil_df.columns = ['Ward', 'Complete', 'Average']
print()
print(sil_df.round(4))

### 3.2 Gráfico comparativo de Silhouette Scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
k_vals = list(K_RANGE)

for method, color, label in zip(LINKAGES, LINKAGE_COLORS, ['Ward', 'Complete', 'Average']):
    scores = results[method]
    best_k = k_vals[scores.index(max(scores))]
    ax.plot(k_vals, scores, marker='o', color=color, linewidth=2, label=label)
    ax.axvline(x=best_k, color=color, linestyle='--', alpha=0.5)
    ax.scatter([best_k], [max(scores)], s=150, color=color, zorder=5)

ax.set_title('Silhouette Score por número de clústers — Clustering Jerárquico', fontsize=13)
ax.set_xlabel('Número de clústers (k)')
ax.set_ylabel('Silhouette Score')
ax.set_xticks(k_vals)
ax.legend(title='Linkage')
plt.tight_layout()
plt.savefig('hier_fig2_silhouette.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig2_silhouette.png')

### 3.3 Dendrograma Ward con línea de corte en k óptimo

In [ ]:
# Determinar k óptimo con Ward
ward_scores = results['ward']
K_OPT = list(K_RANGE)[ward_scores.index(max(ward_scores))]
print(f'K óptimo (Ward): {K_OPT}  |  Silhouette = {max(ward_scores):.4f}')

# Calcular altura de corte para k=K_OPT en Ward
# La altura de corte está entre la fusión número (n-K_OPT) y (n-K_OPT+1)
n = len(X)
cut_height = (Z['ward'][-K_OPT, 2] + Z['ward'][-(K_OPT-1), 2]) / 2
print(f'Altura de corte para k={K_OPT}: {cut_height:.4f}')

fig, ax = plt.subplots(figsize=(14, 7))
dendrogram(
    Z['ward'],
    ax=ax,
    truncate_mode='lastp',
    p=40,
    leaf_rotation=90,
    leaf_font_size=8,
    show_contracted=True,
    color_threshold=cut_height
)
ax.axhline(y=cut_height, color='crimson', linestyle='--', linewidth=2,
           label=f'Línea de corte → k = {K_OPT} clústers')
ax.set_title(f'Dendrograma Ward — Línea de corte en k = {K_OPT} (Silhouette = {max(ward_scores):.4f})', fontsize=13)
ax.set_xlabel('Observaciones (truncado a 40 hojas)')
ax.set_ylabel('Distancia (Varianza Ward)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('hier_fig3_dendrogram_cut.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig3_dendrogram_cut.png')

### 3.4 Asignación de etiquetas con fcluster()

In [ ]:
# Etiquetas sobre la muestra (para visualización y análisis)
labels_sample = fcluster(Z['ward'], K_OPT, criterion='maxclust')

# Etiquetas sobre el dataset completo usando AgglomerativeClustering
agg = AgglomerativeClustering(n_clusters=K_OPT, linkage='ward')
labels_full = agg.fit_predict(X_full) + 1  # 1-based

df_result = df.copy()
df_result['Cluster'] = labels_full

print(f'Distribución de clústers (dataset completo):')
print(df_result['Cluster'].value_counts().sort_index())
print(f'\nTotal: {len(df_result):,} observaciones')

---
## 4. Visualización
### 4.1 PCA 2D

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_full)
var_exp = pca.explained_variance_ratio_ * 100
print(f'Varianza explicada: PC1={var_exp[0]:.1f}%, PC2={var_exp[1]:.1f}%  |  Total={sum(var_exp):.1f}%')

fig, ax = plt.subplots(figsize=(9, 6))
for c in range(1, K_OPT + 1):
    mask = labels_full == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=f'Clúster {c}', alpha=0.45, s=15, color=PALETTE[c-1])

ax.set_title(f'Clustering Jerárquico Ward (k={K_OPT}) — Reducción PCA', fontsize=13)
ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}% varianza)')
ax.legend()
plt.tight_layout()
plt.savefig('hier_fig4_pca.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig4_pca.png')

### 4.2 t-SNE 2D

In [ ]:
N_TSNE = min(5000, len(X_full))
idx_tsne = np.random.choice(len(X_full), N_TSNE, replace=False)
X_tsne_input = X_full[idx_tsne]
labels_tsne = labels_full[idx_tsne]

print(f't-SNE sobre {N_TSNE} muestras...')
tsne = TSNE(n_components=2, perplexity=40, learning_rate='auto',
            init='pca', random_state=RANDOM_STATE, n_jobs=-1)
X_tsne = tsne.fit_transform(X_tsne_input)
print('t-SNE completado ✔')

fig, ax = plt.subplots(figsize=(9, 6))
for c in range(1, K_OPT + 1):
    mask = labels_tsne == c
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               label=f'Clúster {c}', alpha=0.45, s=15, color=PALETTE[c-1])

ax.set_title(f'Clustering Jerárquico Ward (k={K_OPT}) — Reducción t-SNE', fontsize=13)
ax.set_xlabel('Dimensión t-SNE 1')
ax.set_ylabel('Dimensión t-SNE 2')
ax.legend()
plt.tight_layout()
plt.savefig('hier_fig5_tsne.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig5_tsne.png')

### 4.3 Panel PCA vs t-SNE lado a lado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Clustering Jerárquico Ward (k={K_OPT}) — PCA vs t-SNE', fontsize=14)

for c in range(1, K_OPT + 1):
    mask_pca = labels_full == c
    axes[0].scatter(X_pca[mask_pca, 0], X_pca[mask_pca, 1],
                    label=f'Clúster {c}', alpha=0.4, s=12, color=PALETTE[c-1])
    mask_tsne = labels_tsne == c
    axes[1].scatter(X_tsne[mask_tsne, 0], X_tsne[mask_tsne, 1],
                    label=f'Clúster {c}', alpha=0.4, s=12, color=PALETTE[c-1])

axes[0].set_title('PCA (lineal)', fontsize=12)
axes[0].set_xlabel(f'PC1 ({var_exp[0]:.1f}%)')
axes[0].set_ylabel(f'PC2 ({var_exp[1]:.1f}%)')
axes[0].legend()

axes[1].set_title('t-SNE (no-lineal)', fontsize=12)
axes[1].set_xlabel('Dim 1')
axes[1].set_ylabel('Dim 2')
axes[1].legend()

plt.tight_layout()
plt.savefig('hier_fig6_panel_viz.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig6_panel_viz.png')

---
## 5. Análisis e Interpretación
### 5.1 Tabla resumen de clústers

In [ ]:
summary = df_result.groupby('Cluster')[NUM_FEATURES].mean().round(1)
summary['Tamaño'] = df_result.groupby('Cluster').size()
summary['% Total'] = (summary['Tamaño'] / len(df_result) * 100).round(1)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 130)
print('=== Tabla resumen: medias por clúster (Jerárquico Ward) ===')
summary

### 5.2 Heatmap de medias normalizadas

In [ ]:
heat_data = summary[NUM_FEATURES].copy()
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, max(3, K_OPT * 0.8)))
sns.heatmap(heat_norm, annot=heat_data.values, fmt='.0f',
            cmap='Blues', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Valor normalizado'})
ax.set_title(f'Medias por clúster — Jerárquico Ward (k={K_OPT})', fontsize=12)
ax.set_xlabel('Variable')
ax.set_ylabel('Clúster')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('hier_fig7_heatmap.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig7_heatmap.png')

### 5.3 Ejemplos representativos por clúster (tabla)

In [ ]:
REPR_VARS = ['OverallQual', 'GrLivArea', 'GarageArea', 'YearBuilt', 'SalePrice']

print('=== Ejemplos representativos por clúster (3 más cercanos al centroide) ===')
repr_rows = []

for c in range(1, K_OPT + 1):
    mask = df_result['Cluster'] == c
    cluster_data = df_result[mask][REPR_VARS]
    centroid = cluster_data.mean()
    # Distancia de cada punto al centroide
    dists = ((cluster_data - centroid) ** 2).sum(axis=1)
    closest_idx = dists.nsmallest(3).index
    for idx in closest_idx:
        row = df_result.loc[idx, REPR_VARS].to_dict()
        row['Cluster'] = c
        repr_rows.append(row)

repr_df = pd.DataFrame(repr_rows)[['Cluster'] + REPR_VARS]
repr_df = repr_df.sort_values('Cluster').reset_index(drop=True)
repr_df['SalePrice'] = repr_df['SalePrice'].apply(lambda x: f'${x:,.0f}')
repr_df['YearBuilt'] = repr_df['YearBuilt'].astype(int)
repr_df['OverallQual'] = repr_df['OverallQual'].astype(int)
print(repr_df.to_string(index=False))

### 5.4 Interpretación de cada clúster

In [ ]:
print('=== Interpretación de los clústers (Jerárquico Ward) ===')
print()
for c in range(1, K_OPT + 1):
    row = summary.loc[c]
    print(f'Clúster {c}  |  n={int(row["Tamaño"]):,}  ({row["% Total"]:.1f}%)')
    print(f'  SalePrice media:  ${row["SalePrice"]:,.0f}')
    print(f'  GrLivArea media:  {row["GrLivArea"]:.0f} pie²')
    print(f'  OverallQual:      {row["OverallQual"]:.1f}/10')
    print(f'  YearBuilt:        {row["YearBuilt"]:.0f}')
    print(f'  GarageArea:       {row["GarageArea"]:.0f} pie²')
    print()

---
## 6. Comparación con K-Means
### 6.1 Tabla comparativa de Silhouette Scores

In [ ]:
from sklearn.cluster import KMeans

# Recalcular K-Means sobre el mismo dataset limpio (sin outliers)
kmeans_sil = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lbl = km.fit_predict(X_full)
    score = silhouette_score(X_full, lbl, sample_size=3000, random_state=RANDOM_STATE)
    kmeans_sil.append(round(score, 4))

compare_df = pd.DataFrame({
    'k': list(K_RANGE),
    'K-Means': kmeans_sil,
    'Jerárquico Ward': results['ward'],
    'Jerárquico Complete': results['complete'],
    'Jerárquico Average': results['average'],
})
compare_df = compare_df.set_index('k')
print('=== Tabla comparativa de Silhouette Score ===')
print(compare_df.round(4))

### 6.2 Gráfico comparativo K-Means vs Jerárquico

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
k_vals = list(K_RANGE)

styles = [
    ('K-Means',              kmeans_sil,        'crimson',     '-',  'o'),
    ('Jerárquico Ward',      results['ward'],    'steelblue',   '--', 's'),
    ('Jerárquico Complete',  results['complete'],'darkorange',  ':',  '^'),
    ('Jerárquico Average',   results['average'], 'forestgreen', '-.', 'D'),
]

for label, scores, color, ls, marker in styles:
    ax.plot(k_vals, scores, marker=marker, color=color, linestyle=ls,
            linewidth=2, label=label)

ax.set_title('Silhouette Score: K-Means vs Clustering Jerárquico (Ward / Complete / Average)', fontsize=12)
ax.set_xlabel('Número de clústers (k)')
ax.set_ylabel('Silhouette Score')
ax.set_xticks(k_vals)
ax.legend()
plt.tight_layout()
plt.savefig('hier_fig8_comparacion.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig8_comparacion.png')

### 6.3 Visualización lado a lado: K-Means vs Jerárquico (PCA)

In [ ]:
from sklearn.cluster import KMeans

km_opt = KMeans(n_clusters=K_OPT, random_state=RANDOM_STATE, n_init=10)
labels_km = km_opt.fit_predict(X_full) + 1

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Comparación PCA: K-Means vs Jerárquico Ward (k={K_OPT})', fontsize=14)

titles = ['K-Means', 'Jerárquico Ward']
all_labels = [labels_km, labels_full]

for ax, title, lbl in zip(axes, titles, all_labels):
    for c in range(1, K_OPT + 1):
        mask = lbl == c
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   label=f'Clúster {c}', alpha=0.4, s=12, color=PALETTE[c-1])
    ax.set_title(title, fontsize=12)
    ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)')
    ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)')
    ax.legend()

plt.tight_layout()
plt.savefig('hier_fig9_kmeans_vs_hier.png', bbox_inches='tight')
plt.show()
print('Figura guardada: hier_fig9_kmeans_vs_hier.png')

### 6.4 Reflexión comparativa

In [ ]:
best_km_k   = list(K_RANGE)[kmeans_sil.index(max(kmeans_sil))]
best_ward_k = list(K_RANGE)[results['ward'].index(max(results['ward']))]

print('===================================================================')
print('          REFLEXIÓN COMPARATIVA: K-Means vs Jerárquico')
print('===================================================================')
print()
print('1. Número de clústers óptimo:')
print(f'   K-Means:             k = {best_km_k}  (Silhouette = {max(kmeans_sil):.4f})')
print(f'   Jerárquico Ward:     k = {best_ward_k}  (Silhouette = {max(results["ward"]):.4f})')
print(f'   Jerárquico Complete: k = {list(K_RANGE)[results["complete"].index(max(results["complete"]))]}  (Silhouette = {max(results["complete"]):.4f})')
print(f'   Jerárquico Average:  k = {list(K_RANGE)[results["average"].index(max(results["average"]))]}  (Silhouette = {max(results["average"]):.4f})')
print()
print('2. ¿Cambió la forma de los clústers?')
print('   K-Means produce clústers esféricos de similar tamaño (asume igual varianza).')
print('   Ward agrupa minimizando varianza intra-clúster → clústers más compactos')
print('   pero puede generar grupos de tamaño desigual según la jerarquía de fusiones.')
print()
print('3. ¿Se mantuvieron agrupamientos?')
print('   Los segmentos extremos (alto precio vs bajo precio) se mantienen en ambos.')
print('   Las fronteras de clústers intermedios difieren: K-Means es más "suave",')
print('   Ward corta de forma más rígida según las distancias de fusión.')
print()
print('4. ¿Cuál técnica es más adecuada para este caso?')
print('   → K-Means es preferible para producción y segmentación de marketing:')
print('     más eficiente (O(nk)), escalable y fácil de actualizar con nuevos datos.')
print('   → Jerárquico Ward es más adecuado para exploración y cuando no se')
print('     conoce k: el dendrograma revela la estructura natural del mercado')
print('     sin asumir un número fijo de grupos de antemano.')
print('   → Para Ames Housing, ambos coinciden en la estructura general del')
print('     mercado (Económico / Estándar / Calidad), lo que refuerza la')
print('     validez de los segmentos identificados.')